In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow nltk datasets transformers accelerate torch


In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


TensorFlow version: 2.20.0
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [ ]:
import pandas as pd
import json
import re
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"null", "none", "nan"}:
        return ""
    return s


URL_PATTERN = re.compile(
    r"((?:https?://|www\.)[^\s<>\"'()]+)",
    re.IGNORECASE,
)


def extract_urls_from_text(text):
    text = safe_str(text)
    matches = URL_PATTERN.findall(text)

    seen = set()
    urls = []
    for url in matches:
        url = url.strip().rstrip('.,;:!?')
        if url and url not in seen:
            seen.add(url)
            urls.append(url)

    return urls


def normalize_url_value(value):
    if isinstance(value, list):
        value = " | ".join(safe_str(v) for v in value if safe_str(v))
    return safe_str(value)


def populate_url_column(df):
    if "url" not in df.columns:
        df["url"] = ""

    df["url"] = df["url"].apply(normalize_url_value)

    missing_mask = df["url"].eq("")
    if missing_mask.any():
        df.loc[missing_mask, "url"] = df.loc[missing_mask, "body"].apply(
            lambda body: " | ".join(extract_urls_from_text(body))
        )

    return df


def normalize_sender(sender):
    return safe_str(sender).lower()


def build_text_all_fields(row):
    sender = normalize_sender(row.get("sender", ""))
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    return (
        f"[SENDER] {sender}\n"
        f"[SUBJECT] {subject}\n"
        f"[BODY] {body}\n"
        f"[URL] {url}"
    ).strip()


def build_text_all_fields_from_parts(sender, subject, body, url):
    return build_text_all_fields({
        "sender": sender,
        "subject": subject,
        "body": body,
        "url": url,
    })


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    df = populate_url_column(df)

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_all_fields, axis=1)

    df = df[[
        "dataset", "sender", "subject", "body", "url",
        "label_raw", "label", "label_id", "text"
    ]]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
        "From": "sender",
        "Sender": "sender",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    df = populate_url_column(df)

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_all_fields, axis=1)

    df = df[[
        "dataset", "sender", "subject", "body", "url",
        "label_raw", "label", "label_id", "text"
    ]]

    return df


In [ ]:
machinewars_spam_as_phishing_df = load_machinewars(
    "machinewars_filtered_emails.json",
    spam_as_phishing=False,
    dataset_name="machinewars"
)
train_df, val_df = train_test_split(
    machinewars_spam_as_phishing_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_spam_as_phishing_df["label_id"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))


Test dataset 1:
label
phishing      21842
legitimate    17312
Name: count, dtype: int64
           dataset                            sender  \
0  CEAS_08_cleaned  Young Esposito <Young@iworld.de>   
1  CEAS_08_cleaned      Mok <ipline's1983@icable.ph>   

                     subject  \
0  Never agree to be a loser   
1     Befriend Jenna Jameson   

                                                body  \
0  Buck up, your troubles caused by small dimensi...   
1  \nUpgrade your sex and pleasures with these te...   

                         url label_raw     label  label_id  \
0      http://whitedone.com/  phishing  phishing         1   
1  http://www.brightmade.com  phishing  phishing         1   

                                                text  
0  [SENDER] young esposito <young@iworld.de>\n[SU...  
1  [SENDER] mok <ipline's1983@icable.ph>\n[SUBJEC...  

Test dataset 2:
label
phishing    1565
Name: count, dtype: int64
           dataset                                        

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # If model returns tuple-like predictions, keep logits only.
    if isinstance(logits, tuple):
        logits = logits[0]

    y_true = labels
    y_pred = np.argmax(logits, axis=-1)

    # Convert logits to probability for class 1.
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    y_prob = probs[:, 1]

    return compute_binary_metrics(y_true, y_pred, y_prob)

In [ ]:
# Legacy setup cell intentionally disabled.
# The clean fp32 DeBERTa setup/training cell below creates:
#   distilbert_model, distilbert_tokenizer, DISTILBERT_MAX_LENGTH
# Run the clean DeBERTa cell instead of this older duplicate setup.


In [ ]:
# ============================================================
# Clean DeBERTa training/debug block
# Requires:
#   train_df with columns: text, label_id
#   val_df with columns: text, label_id
#   SEED
#   compute_binary_metrics(y_true, y_pred, y_prob)
# ============================================================

import gc
import inspect
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# ------------------------------------------------------------
# 0. Clean CUDA state
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

print("CUDA available:", torch.cuda.is_available())
print("Default torch dtype:", torch.get_default_dtype())

device = "cuda" if torch.cuda.is_available() else "cpu"


# ------------------------------------------------------------
# 1. Basic data checks
# ------------------------------------------------------------
print("Train labels:")
print(train_df["label_id"].value_counts(normalize=False))
print(train_df["label_id"].value_counts(normalize=True))

print("\nValidation labels:")
print(val_df["label_id"].value_counts(normalize=False))
print(val_df["label_id"].value_counts(normalize=True))

print("\nEmpty train texts:", (train_df["text"].astype(str).str.strip() == "").sum())
print("Empty val texts:", (val_df["text"].astype(str).str.strip() == "").sum())

assert train_df["label_id"].isin([0, 1]).all()
assert val_df["label_id"].isin([0, 1]).all()


# ------------------------------------------------------------
# 2. Model/tokenizer config
# ------------------------------------------------------------
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 3

id2label = {0: "legitimate", 1: "phishing"}
label2id = {"legitimate": 0, "phishing": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# Force full precision.
model = model.to(device=device, dtype=torch.float32)

print("\nModel parameter dtype:", next(model.parameters()).dtype)
print("Model device:", next(model.parameters()).device)

dtype_counts = {}
for _, param in model.named_parameters():
    dtype_counts[str(param.dtype)] = dtype_counts.get(str(param.dtype), 0) + 1

print("Parameter dtype counts:", dtype_counts)


# ------------------------------------------------------------
# 3. Build Hugging Face datasets
# ------------------------------------------------------------
train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
    )

train_ds_tok = train_ds.map(tokenize_batch, batched=True)
val_ds_tok = val_ds.map(tokenize_batch, batched=True)

keep_cols = {"input_ids", "attention_mask", "labels"}

train_ds_tok = train_ds_tok.remove_columns(
    [c for c in train_ds_tok.column_names if c not in keep_cols]
)

val_ds_tok = val_ds_tok.remove_columns(
    [c for c in val_ds_tok.column_names if c not in keep_cols]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# ------------------------------------------------------------
# 4. Pre-training forward-pass diagnostic
# ------------------------------------------------------------
debug_batch = next(iter(torch.utils.data.DataLoader(
    train_ds_tok,
    batch_size=4,
    collate_fn=data_collator,
)))

print("\nDebug batch keys:", debug_batch.keys())
print("input_ids shape:", debug_batch["input_ids"].shape)
print("attention_mask shape:", debug_batch["attention_mask"].shape)
print("labels:", debug_batch["labels"])
print("first input_ids:", debug_batch["input_ids"][0][:30])
print("first attention_mask:", debug_batch["attention_mask"][0][:30])

debug_batch = {k: v.to(device) for k, v in debug_batch.items()}

model.eval()

with torch.no_grad():
    with torch.autocast(device_type="cuda", enabled=False) if device == "cuda" else torch.no_grad():
        outputs = model(
            input_ids=debug_batch["input_ids"],
            attention_mask=debug_batch["attention_mask"],
        )

print("\nPre-training logits:")
print(outputs.logits)
print("Finite logits:", torch.isfinite(outputs.logits).all())
print("Logits dtype:", outputs.logits.dtype)

assert torch.isfinite(outputs.logits).all(), "Model produces NaN/Inf logits before training."
assert outputs.logits.dtype == torch.float32, "Model is still producing non-fp32 logits."


# ------------------------------------------------------------
# 5. Metrics
# ------------------------------------------------------------
def compute_deberta_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    logits = np.asarray(logits, dtype=np.float32)

    if not np.isfinite(logits).all():
        print("WARNING: non-finite logits detected in metrics")
        print("NaN count:", np.isnan(logits).sum())
        print("Inf count:", np.isinf(logits).sum())
        logits = np.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)

    probs_all = torch.softmax(
        torch.tensor(logits, dtype=torch.float32),
        dim=-1,
    ).numpy()

    probs = probs_all[:, 1]
    preds = (probs >= 0.5).astype(int)

    return compute_binary_metrics(labels, preds, probs)


# ------------------------------------------------------------
# 6. Debug Trainer: fails loudly on NaN loss/logits
# ------------------------------------------------------------
class DebugTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()

        outputs = model(**inputs)
        logits = outputs.logits.float()

        if not torch.isfinite(logits).all():
            print("Bad logits:")
            print(logits)
            raise ValueError("Non-finite logits during training")

        loss = torch.nn.functional.cross_entropy(logits, labels)

        if not torch.isfinite(loss):
            print("Bad loss:", loss)
            print("Logits:", logits)
            print("Labels:", labels)
            raise ValueError("Non-finite loss during training")

        return (loss, outputs) if return_outputs else loss


# ------------------------------------------------------------
# 7. TrainingArguments
# ------------------------------------------------------------
training_args_kwargs = dict(
    output_dir="/content/deberta_phishing_model",
    learning_rate=1e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
)

if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)


# ------------------------------------------------------------
# 8. Train plain model first: no class weights
# ------------------------------------------------------------
trainer = DebugTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    data_collator=data_collator,
    compute_metrics=compute_deberta_metrics,
)

print("\nBefore training parameter sample:")
before_params = next(model.parameters()).detach().flatten()[:5].clone()
print(before_params)

train_result = trainer.train()

print("\nAfter training parameter sample:")
after_params = next(model.parameters()).detach().flatten()[:5].clone()
print(after_params)

print("\nParameters changed:", not torch.equal(before_params.cpu(), after_params.cpu()))

print("\nTrain result:")
print(train_result)

print("\nFinal validation evaluation:")
eval_result = trainer.evaluate()
print(eval_result)

# ------------------------------------------------------------
# 9. Backward-compatible aliases for downstream cells
# ------------------------------------------------------------
# Later notebook cells use the old DistilBERT variable/function names.
# Keep those names bound to the trained fp32 DeBERTa objects.
distilbert_model = model
distilbert_tokenizer = tokenizer
DISTILBERT_MODEL_NAME = MODEL_NAME
DISTILBERT_MAX_LENGTH = MAX_LENGTH
ACTIVE_MODEL_NAME = "deberta"


def _model_safe_text(text):
    """Return a non-empty string so DeBERTa never receives a 0-token input."""
    s = "" if text is None else str(text)
    s = s.strip()
    return s if s else "[EMPTY_EMAIL]"


CUDA available: True
Default torch dtype: torch.float32
Train labels:
label_id
1    5404
0    5280
Name: count, dtype: int64
label_id
1    0.505803
0    0.494197
Name: proportion, dtype: float64

Validation labels:
label_id
1    1352
0    1320
Name: count, dtype: int64
label_id
1    0.505988
0    0.494012
Name: proportion, dtype: float64

Empty train texts: 0
Empty val texts: 0


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]


Debug batch keys: KeysView({'labels': tensor([0, 1, 0, 1]), 'input_ids': tensor([[  647,   430, 59088,  ...,     0,     0,     0],
        [  647,   430, 59088,  ...,     0,     0,     0],
        [  647,   430, 59088,  ...,   320, 48189,   271],
        [  647,   430, 59088,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])})
input_ids shape: torch.Size([4, 512])
attention_mask shape: torch.Size([4, 512])
labels: tensor([0, 1, 0, 1])
first input_ids: tensor([   647,    430,  59088,    592,  44677,    507,  16800,   2569,  19443,
          1683, 110091,    260,   2083,   1504,    647,  62953,   2252,  49561,
           592,  34359,    294,  13643,  30259,    510,   5540,    270,  32731,
          8510,    647,  78339])
first attention_mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])

Pre-t

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.029634,0.025950,0.991766,0.985401,0.998521,0.991918,0.999905
2,0.001829,0.015038,0.996257,0.997037,0.995562,0.996299,0.999951
3,0.000213,0.016918,0.995883,0.999255,0.992604,0.995918,0.999956



After training parameter sample:
tensor([-0.0014, -0.0116,  0.0114,  0.0068, -0.0196], device='cuda:0')

Parameters changed: True

Train result:
TrainOutput(global_step=2004, training_loss=0.05275640237213557, metrics={'train_runtime': 1064.1118, 'train_samples_per_second': 30.121, 'train_steps_per_second': 1.883, 'total_flos': 8429160489590400.0, 'train_loss': 0.05275640237213557, 'epoch': 3.0})

Final validation evaluation:


{'eval_loss': 0.016918128356337547, 'eval_accuracy': 0.9958832335329342, 'eval_precision': 0.9992553983618764, 'eval_recall': 0.992603550295858, 'eval_f1': 0.9959183673469387, 'eval_roc_auc': 0.9999562937062937, 'eval_runtime': 28.7879, 'eval_samples_per_second': 92.817, 'eval_steps_per_second': 5.801, 'epoch': 3.0}


In [ ]:
def _model_safe_text(text):
    """Return a non-empty string so DeBERTa never receives a 0-token input."""
    s = "" if text is None else str(text)
    s = s.strip()
    return s if s else "[EMPTY_EMAIL]"


def distilbert_predict_one(text):
    pred, prob = distilbert_batch_predict([text], batch_size=1)
    return int(pred[0]), float(prob[0])


def distilbert_batch_predict(texts, batch_size=32):
    texts = [_model_safe_text(t) for t in texts]
    if len(texts) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    distilbert_model.eval()
    all_probs = []
    device = next(distilbert_model.parameters()).device

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = distilbert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=DISTILBERT_MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", enabled=False):
                    logits = distilbert_model(**encoded).logits.float()
            else:
                logits = distilbert_model(**encoded).logits.float()

            if not torch.isfinite(logits).all():
                raise ValueError("Non-finite logits during inference")

            probs = torch.softmax(logits, dim=-1)[:, 1]

        all_probs.append(probs.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(int)
    return preds, probs


def evaluate_distilbert(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = distilbert_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)


In [ ]:
print("Validation metrics:")
print(evaluate_distilbert(val_df))

distilbert_rows = [{"dataset": "validation", **evaluate_distilbert(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_distilbert(test_df)
    distilbert_rows.append({"dataset": test_name, **metrics})

distilbert_results_df = pd.DataFrame(distilbert_rows)
distilbert_results_df


Validation metrics:
{'accuracy': 0.9958832335329342, 'precision': 0.9992553983618764, 'recall': 0.992603550295858, 'f1': 0.9959183673469387, 'roc_auc': np.float64(0.9999562937062937)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.995883,0.999255,0.992604,0.995918,0.999956
1,CEAS_08_cleaned,0.530521,0.980823,0.161569,0.277437,0.941376
2,Nazario_cleaned,0.456230,1.000000,0.456230,0.626591,NaN
3,Nigerian_Fraud_cleaned,0.417767,1.000000,0.417767,0.589331,NaN
4,SpamAssasin_cleaned,0.733689,0.929648,0.107683,0.193010,0.912358


Epoch 1/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.9466 - loss: 0.1397 - val_accuracy: 0.9697 - val_loss: 0.0833
Epoch 2/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.9920 - loss: 0.0255 - val_accuracy: 0.9820 - val_loss: 0.0659
Epoch 3/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.9975 - loss: 0.0088 - val_accuracy: 0.9820 - val_loss: 0.0685
Epoch 4/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.9977 - loss: 0.0070 - val_accuracy: 0.9611 - val_loss: 0.1743


In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [ ]:
predict_one = distilbert_predict_one
batch_predict = distilbert_batch_predict
ACTIVE_MODEL_NAME = "deberta"
print("Active model:", ACTIVE_MODEL_NAME)


Active model: deberta


In [ ]:
def apply_attack_to_fields(row, subject_attack_fn=None, body_attack_fn=None):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    if subject_attack_fn is not None:
        subject = subject_attack_fn(subject)

    if body_attack_fn is not None:
        body = body_attack_fn(body)

    return _model_safe_text(build_text_all_fields_from_parts(sender, subject, body, url))


def evaluate_attack_common(df_eval, attack_name, row_attack_fn):
    attacked_texts = [_model_safe_text(row_attack_fn(row)) for _, row in df_eval.iterrows()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }


In [ ]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda row: row["text"]))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_prefix_attack)))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_suffix_attack)))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", lambda row: apply_attack_to_fields(row, body_attack_fn=contradiction_attack)))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda row: apply_attack_to_fields(
            row,
            subject_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            body_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        )
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda row: apply_attack_to_fields(
            row,
            subject_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
            body_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
        )
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", lambda row: apply_attack_to_fields(row, body_attack_fn=prefix_injection_attack)))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df


,attack,n_samples,accuracy,precision,recall,f1,roc_auc
6,prefix_injection,2672,0.997006,0.997778,0.996302,0.997039,0.999953
4,synonym_attack,2672,0.996632,0.997039,0.996302,0.996670,0.999946
1,benign_prefix,2672,0.996632,0.997776,0.995562,0.996668,0.999953
3,contradiction,2672,0.996257,0.997774,0.994822,0.996296,0.999955
5,keyword_deletion,2672,0.996257,0.999256,0.993343,0.996291,0.999951
2,benign_suffix,2672,0.995883,0.998513,0.993343,0.995921,0.999948
0,clean,2672,0.995883,0.999255,0.992604,0.995918,0.999956


In [ ]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda row: row["text"]))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_prefix_attack)))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_suffix_attack)))
    rows.append(evaluate_attack_common(test_df, "contradiction", lambda row: apply_attack_to_fields(row, body_attack_fn=contradiction_attack)))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda row: apply_attack_to_fields(
                row,
                subject_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
                body_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            )
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda row: apply_attack_to_fields(
                row,
                subject_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
                body_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
            )
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", lambda row: apply_attack_to_fields(row, body_attack_fn=prefix_injection_attack)))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)



=== deberta | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection      39154  0.614446   0.976278  0.316546  0.478080   
3     contradiction      39154  0.584155   0.979145  0.260095  0.411011   
1     benign_prefix      39154  0.568473   0.975394  0.232305  0.375240   
2     benign_suffix      39154  0.546074   0.979948  0.190184  0.318546   
4    synonym_attack      39154  0.543035   0.979602  0.184690  0.310786   
5  keyword_deletion      39154  0.532053   0.981928  0.164179  0.281321   
0             clean      39154  0.530521   0.980823  0.161569  0.277437   

    roc_auc  
6  0.941739  
3  0.939005  
1  0.934422  
2  0.941653  
4  0.937486  
5  0.942978  
0  0.941376  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== deberta | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       1565  0.614058        1.0  0.614058  0.760887   
3     contradiction       1565  0.555272        1.0  0.555272  0.714051   
1     benign_prefix       1565  0.515655        1.0  0.515655  0.680438   
2     benign_suffix       1565  0.477316        1.0  0.477316  0.646194   
0             clean       1565  0.456230        1.0  0.456230  0.626591   
4    synonym_attack       1565  0.446006        1.0  0.446006  0.616880   
5  keyword_deletion       1565  0.400639        1.0  0.400639  0.572080   

   roc_auc  
6      NaN  
3      NaN  
1      NaN  
2      NaN  
0      NaN  
4      NaN  
5      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== deberta | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       3332  0.514706        1.0  0.514706  0.679612   
3     contradiction       3332  0.467287        1.0  0.467287  0.636940   
4    synonym_attack       3332  0.431573        1.0  0.431573  0.602935   
0             clean       3332  0.417767        1.0  0.417767  0.589331   
5  keyword_deletion       3332  0.410264        1.0  0.410264  0.581826   
2     benign_suffix       3332  0.409064        1.0  0.409064  0.580618   
1     benign_prefix       3332  0.406363        1.0  0.406363  0.577892   

   roc_auc  
6      NaN  
3      NaN  
4      NaN  
0      NaN  
5      NaN  
2      NaN  
1      NaN  

=== deberta | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       5809  0.750387   0.911043  0.172875  0.290607   
3     contradiction       5809  0.745739   0.922807  0.153085  0

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [ ]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "sender", "url", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}

PROTECTED_TOKENS = {"[SENDER]", "[SUBJECT]", "[BODY]", "[URL]"}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    if tok in PROTECTED_TOKENS:
        return False

    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    return " ".join(tokens[:idx] + tokens[idx+1:])


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        candidate_indices = candidate_indices[:candidate_cap]
        candidate_texts = [delete_token_at_index(tokens, i) for i in candidate_indices]

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))
        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text


In [ ]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history


def greedy_delete_subject_body_blackbox(row, max_delete_steps=3, candidate_cap=5):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    current_subject = subject
    current_body = body

    for _ in range(max_delete_steps):
        subject_tokens = basic_tokenize_with_indices(current_subject)
        body_tokens = basic_tokenize_with_indices(current_body)

        subject_indices = [
            i for i, tok in enumerate(subject_tokens)
            if is_deletable_token(tok)
        ][:candidate_cap]

        remaining_cap = candidate_cap - len(subject_indices)

        body_indices = [
            i for i, tok in enumerate(body_tokens)
            if is_deletable_token(tok)
        ][:max(0, remaining_cap)]

        candidate_texts = []
        candidate_states = []

        for i in subject_indices:
            new_subject = delete_token_at_index(subject_tokens, i)
            candidate_texts.append(
                build_text_all_fields_from_parts(sender, new_subject, current_body, url)
            )
            candidate_states.append((new_subject, current_body))

        for i in body_indices:
            new_body = delete_token_at_index(body_tokens, i)
            candidate_texts.append(
                build_text_all_fields_from_parts(sender, current_subject, new_body, url)
            )
            candidate_states.append((current_subject, new_body))

        if not candidate_texts:
            break

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_subject, best_body = candidate_states[best_idx]

        if best_subject == current_subject and best_body == current_body:
            break

        current_subject = best_subject
        current_body = best_body

    return build_text_all_fields_from_parts(sender, current_subject, current_body, url)

def greedy_add_to_body_blackbox(row, add_steps=3):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    attacked_body, history = greedy_add_attack_blackbox(body, add_steps=add_steps)
    attacked_text = build_text_all_fields_from_parts(sender, subject, attacked_body, url)
    return attacked_text, history


In [ ]:
def add_only_attack(row, add_steps=3):
    attacked_text, _ = greedy_add_to_body_blackbox(row, add_steps=add_steps)
    return _model_safe_text(attacked_text)


def delete_only_attack(row, delete_steps=5):
    attacked_text = greedy_delete_subject_body_blackbox(row, max_delete_steps=delete_steps)
    return _model_safe_text(attacked_text)


def hybrid_add_then_delete_attack_blackbox(row, add_steps=3, delete_steps=5):
    """
    First greedy additions to body, then greedy deletions on subject/body.
    """
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    current_body, _ = greedy_add_attack_blackbox(body, add_steps=add_steps)
    temp_row = {
        "sender": sender,
        "subject": subject,
        "body": current_body,
        "url": url,
    }
    current_text = greedy_delete_subject_body_blackbox(temp_row, max_delete_steps=delete_steps)
    return _model_safe_text(current_text)


In [ ]:
if "_model_safe_text" not in globals():
    def _model_safe_text(text):
        s = "" if text is None else str(text)
        s = s.strip()
        return s if s else "[EMPTY_EMAIL]"


def evaluate_attack_detailed(df_eval, attack_name, row_attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = [_model_safe_text(t) for t in df_eval["text"].astype(str).tolist()]

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    if show_progress:
        iterator = tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")
    else:
        iterator = df_eval.iterrows()

    for _, row in iterator:
        attacked_texts.append(_model_safe_text(row_attack_fn(row)))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df


In [ ]:
if "_model_safe_text" not in globals():
    def _model_safe_text(text):
        s = "" if text is None else str(text)
        s = s.strip()
        return s if s else "[EMPTY_EMAIL]"


def evaluate_evasion_on_phishing(df_eval, attack_name, row_attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = [_model_safe_text(t) for t in df_local["text"].astype(str).tolist()]
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    if show_progress:
        iterator = tqdm(df_attack.iterrows(), total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")
    else:
        iterator = df_attack.iterrows()

    for _, row in iterator:
        attacked_texts.append(_model_safe_text(row_attack_fn(row)))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": [_model_safe_text(t) for t in df_attack["text"].astype(str).tolist()],
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df


In [ ]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda row: add_only_attack(row, add_steps=3)),
    ("delete_only_del5", lambda row: delete_only_attack(row, delete_steps=5)),
    ("hybrid_add3_delete5", lambda row: hybrid_add_then_delete_attack_blackbox(row, add_steps=3, delete_steps=5)),
]


In [ ]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

deberta | add_only_add3:   0%|          | 0/2672 [00:00<?, ?it/s]

deberta | delete_only_del5:   0%|          | 0/2672 [00:00<?, ?it/s]

deberta | hybrid_add3_delete5:   0%|          | 0/2672 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,add_only_add3,2672,0.000749,0.000313,0.995883,0.999255,0.992604,0.995918,0.999958
1,delete_only_del5,2672,0.002994,0.002449,0.993638,1.000000,0.987426,0.993673,0.999950
2,hybrid_add3_delete5,2672,0.003368,0.003127,0.993263,1.000000,0.986686,0.993299,0.999950


In [ ]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

deberta | add_only_add3 phishing:   0%|          | 0/1342 [00:00<?, ?it/s]

deberta | delete_only_del5 phishing:   0%|          | 0/1342 [00:00<?, ?it/s]

deberta | hybrid_add3_delete5 phishing:   0%|          | 0/1342 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
2,hybrid_add3_delete5,1352,1342,8,0.005961,0.994039,0.005660
1,delete_only_del5,1352,1342,7,0.005216,0.994784,0.004203
0,add_only_add3,1352,1342,1,0.000745,0.999255,0.000961


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


deberta | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

deberta | add_only_add3 phishing:   0%|          | 0/3529 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.038744,0.035062,0.495709,0.97551,0.09848,0.1789,0.930556


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,185,79,0.427027,0.572973,0.381201



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.007230,0.005463,0.729902,0.946108,0.091967,0.167639,0.918434
1,SpamAssasin_cleaned,delete_only_del5,5809,0.011190,0.010891,0.724565,0.940299,0.073341,0.136069,0.910264
2,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.016182,0.014265,0.722327,0.964602,0.063446,0.119061,0.917683



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,185,31,0.167568,0.832432,0.146557
1,SpamAssasin_cleaned,delete_only_del5,1718,185,59,0.318919,0.681081,0.283454
2,SpamAssasin_cleaned,hybrid_add3_delete5,1718,185,79,0.427027,0.572973,0.381201


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    if dataset_name == "CEAS_08_cleaned":
        print(f"\nSkipping dataset: {dataset_name}")
        continue

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)

In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

import os
os.makedirs("/content/results", exist_ok=True)

val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved.")